In [27]:
import pandas as pd
import requests
import json
import time

In [28]:
from openai import OpenAI


In [ ]:
client = OpenAI(
    api_key="Enter Your API Key Here",
    base_url="https://api.kluster.ai/v1"
)

In [30]:
df = pd.read_csv("R1_input_data.csv")
df

,input
0,Dice is the leading career destination for tec...
1,"In a world of possibilities, pursue one with e..."
2,About the job\nAbout Rocket Lawyer\n\nWe belie...
3,About the job\nThis role is with Maximus. WayU...
4,About the job\nXometry (NASDAQ: XMTR) powers t...
...,...
1020,Dimensional is a privately owned global invest...
1021,Sr OR & Advanced Analytics Specialist I (Data ...
1022,"**Portfolio Manager, Information Capital and D..."
1023,"As a Data Architect, your role will be to tran..."


In [31]:
# df = df[:4]


In [32]:
df

,input
0,Dice is the leading career destination for tec...
1,"In a world of possibilities, pursue one with e..."
2,About the job\nAbout Rocket Lawyer\n\nWe belie...
3,About the job\nThis role is with Maximus. WayU...
4,About the job\nXometry (NASDAQ: XMTR) powers t...
...,...
1020,Dimensional is a privately owned global invest...
1021,Sr OR & Advanced Analytics Specialist I (Data ...
1022,"**Portfolio Manager, Information Capital and D..."
1023,"As a Data Architect, your role will be to tran..."


In [33]:
def reasoning_prompt(job_description):
    return f"""
You are an experienced skill extraction model. Your task is to extract skills from the given job description using a 4-step reasoning process. Be concise and precise in your reasoning.

Step 1: Understand the Role and Context
Describe how the job title fits into the company’s structure or industry. Consider the function, domain, and purpose of the role.

Step 2: Extract Explicit Skills
List all tools, technologies, certifications, and named skills that are clearly and directly mentioned in the job description.
For each skill, provide a short reason explaining why it was extracted.

Step 3: Infer Implicit Skills
Using the context from Step 1 and the explicit skills from Step 2, infer additional skills that are not directly stated but are clearly implied by the responsibilities or expectations.
For each inferred skill, provide a short reason tied to the job description or context.

Step 4: Thinking Log
show your detailed reasoning process for each step above. This will be used to help train smaller models to mimic your thinking.

Output format:
## Thinking
step 1: ...
step 2: ...
step 3: ...

## Skills
skill 1(implicit): reason 1
skill 2(explicit): reason 2
...

Begin analysis using the job description below:

\"\"\"
{job_description}
\"\"\"
"""

In [34]:
import re

def clean_output(output_text):
    """
    Removes the <think>...</think> block from the output string.
    Returns the cleaned JSON string.
    """
    # Regex pattern to match <think> ... </think> including newlines
    think_pattern = re.compile(r"<think>.*?</think>", re.DOTALL)

    # Remove the think block
    cleaned_text = re.sub(think_pattern, "", output_text)

    # Strip whitespace
    return cleaned_text.strip()